# The Call — AI Commentator Studio (Streamlit Instructor Notebook)

A hands-on Generative AI project for students. Build and run an interactive sports commentator in Google Colab.

**The visitor-facing app has only four inputs:**
1. Choose sport — Cricket or Football
2. Commentary delivery — Energetic, Dramatic, Funny, Emotional or Professional
3. Language — English, Urdu or Roman Urdu
4. Describe the match situation

The app creates commentary text with **Groq inference**, then automatically creates spoken narration with **ElevenLabs TTS** when configured. If ElevenLabs voice is unavailable or rate-limited, a browser voice fallback is provided.

**Important:** This project uses already-trained hosted models. It does not train or download a model. Keep API keys in Colab Secrets. The Streamlit preview runs inside your active Colab runtime; disconnecting or stopping the runtime stops the app.


## 1. Upload and unpack the Streamlit project

Download `AI_Commentator_Studio_Streamlit_ElevenLabs.zip` and upload it when prompted. The archive should contain an `AI_Commentator_Studio/` folder.


In [ ]:
from google.colab import files
from pathlib import Path
from zipfile import ZipFile
import shutil

uploaded = files.upload()
zip_names = [name for name in uploaded if name.lower().endswith(".zip")]
if not zip_names:
    raise RuntimeError("Please upload AI_Commentator_Studio_Streamlit_ElevenLabs.zip")

zip_path = Path(zip_names[0])
project_root = Path("/content")
project_dir = project_root / "AI_Commentator_Studio"

# Remove only the previous extracted copy so files cannot be mixed across versions.
if project_dir.exists():
    shutil.rmtree(project_dir)

with ZipFile(zip_path) as archive:
    archive.extractall(project_root)

required_files = [
    "app.py", "commentary_engine.py", "requirements.txt",
    "app.css", "stage.css", "_characters.py"
]
missing = [name for name in required_files if not (project_dir / name).exists()]
if missing:
    raise FileNotFoundError(f"The uploaded ZIP is incomplete. Missing: {missing}")

print("Project extracted:", project_dir)
print("Verified files:", ", ".join(required_files))

## 2. Install the application libraries

This installs the Streamlit interface, Groq client and Requests. ElevenLabs is called through its official HTTPS API, so no additional TTS SDK is required. This does not download an AI model. CSS and character artwork are separate project files; the notebook contains no CSS injection or search-and-replace patches.


In [ ]:
%pip -q install -r /content/AI_Commentator_Studio/requirements.txt
print("Application dependencies installed.")

## 3. Configure API keys securely

In the Colab left sidebar, open the **key icon → Secrets** and add:

- `GROQ_API_KEY` — required for live commentary.
- `ELEVENLABS_API_KEY` — optional, but enables expressive generated narration.

Turn **Notebook access** on for each secret. Never paste secret values into code cells or print them.


In [ ]:
from google.colab import userdata
import os
import sys


def read_secret(name):
    try:
        return (userdata.get(name) or "").strip()
    except Exception:
        return ""

GROQ_API_KEY = read_secret("GROQ_API_KEY")
ELEVENLABS_API_KEY = read_secret("ELEVENLABS_API_KEY")

if GROQ_API_KEY:
    os.environ["GROQ_API_KEY"] = GROQ_API_KEY
if ELEVENLABS_API_KEY:
    os.environ["ELEVENLABS_API_KEY"] = ELEVENLABS_API_KEY
os.environ["GROQ_MODEL"] = "openai/gpt-oss-20b"
os.environ["ELEVENLABS_TTS_MODEL"] = "eleven_v4"

# Remove any earlier imported engine from a previous project revision.
sys.modules.pop("commentary_engine", None)
if str(project_dir) not in sys.path:
    sys.path.insert(0, str(project_dir))

print("Groq configured:", bool(GROQ_API_KEY))
print("ElevenLabs TTS configured:", bool(ELEVENLABS_API_KEY))
print("Secrets are not displayed.")


## 4. Test the Groq API

This is a small live inference test. GPT-OSS uses `reasoning_effort="low"` and `include_reasoning=False`. If the key is missing, this cell skips the live request; the offline fallback can still be tested later.

In [ ]:
if not GROQ_API_KEY:
    print("Skipped live Groq request. Add GROQ_API_KEY in Colab Secrets to enable it.")
else:
    from groq import Groq
    client = Groq(api_key=GROQ_API_KEY.strip(), timeout=60.0, max_retries=1)
    response = client.chat.completions.create(
        model="openai/gpt-oss-20b",
        messages=[{
            "role": "user",
            "content": (
                "You are an original energetic cricket commentator. "
                "A batter has hit a confirmed six over long-on. "
                "Write two short, exciting broadcast sentences. "
                "Do not invent a player name or score. Return only the commentary."
            ),
        }],
        reasoning_effort="low",
        include_reasoning=False,
        max_completion_tokens=1024,
        temperature=0.7,
        stream=False,
    )
    test_text = (response.choices[0].message.content or "").strip()
    if test_text:
        print("Groq API test successful:\n")
        print(test_text)
    else:
        print("Groq returned an empty final message.")
        print("Finish reason:", response.choices[0].finish_reason)
        print("Completion tokens:", response.usage.completion_tokens)

## 5. Import and inspect the commentary engine

`commentary_engine.py` contains the style-specific prompt instructions, event detection, safe factual constraints, offline fallback, Groq generation and ElevenLabs TTS integration.


In [ ]:
import importlib
import commentary_engine as engine

importlib.reload(engine)
print("Groq model:", engine.GROQ_MODEL)
print("ElevenLabs TTS model:", engine.ELEVENLABS_TTS_MODEL)
print("Default voice ID:", engine.ELEVENLABS_DEFAULT_VOICE_ID)
print("Sports:", engine.SPORTS)
print("Delivery styles:", engine.STYLES)
print("Languages:", engine.LANGUAGES)


## 6. Generate a sample commentary

This calls the same backend function used by the Streamlit application. It uses the supplied situation as the source of facts and does not invent names, scores or outcomes.

In [ ]:
sample_commentary, sample_status, sample_fallback, sample_event = engine.generate_commentary(
    sport="Cricket",
    situation="The batter swings through the line and hits a clean six over long-on.",
    style="Energetic",
    language="English",
    personality=(
        "A charismatic, expressive, witty and fact-grounded original stadium commentator. "
        "Use natural live-broadcast phrasing and celebrate confirmed action."
    ),
    commentator_name="The Boundary Voice",
    event_choice="Auto-detect",
)

print("Status:", sample_status)
print("Detected event:", sample_event)
print("Fallback used:", sample_fallback)
print("\nCOMMENTARY:\n", sample_commentary)

## 7. Generate expressive ElevenLabs voice

ElevenLabs v4 receives style-specific audio tags and per-request voice settings. Energetic, Dramatic, Funny, Emotional and Professional use different delivery directions. If the ElevenLabs key is not configured or the service is rate-limited, the final app offers device/browser narration instead. The API returns MP3 audio for the Streamlit player.


In [ ]:
sample_audio = None
sample_audio_status = "ElevenLabs key not configured; browser narration will be used by the app."

if sample_commentary and ELEVENLABS_API_KEY:
    import base64
    from IPython.display import HTML, display
    sample_audio, sample_audio_status = engine.synthesize_expressive_audio(
        text=sample_commentary,
        style="Energetic",
        language="English",
        voice_id="",
        intensity=4,
    )
    print("Voice status:", sample_audio_status)
    if sample_audio:
        encoded_audio = base64.b64encode(sample_audio).decode("ascii")
        display(HTML(
            f'<audio controls preload="metadata" src="data:audio/mpeg;base64,{encoded_audio}"></audio>'
        ))
        with open("ai_commentator_elevenlabs_test.mp3", "wb") as audio_file:
            audio_file.write(sample_audio)
        print("MP3 saved as ai_commentator_elevenlabs_test.mp3")
else:
    print(sample_audio_status)


## 8. Run offline regression tests

These tests do not spend API credits. They verify the text-generation parameters, event detection, five distinct styles, Urdu/Roman Urdu output, ElevenLabs style tags and MP3 request handling, plus the animated Cricket/Football stage.


In [ ]:
import subprocess
import sys

project_dir = Path("/content/AI_Commentator_Studio")
test_result = subprocess.run(
    [sys.executable, "-m", "unittest", "discover", "-s", "tests", "-v"],
    cwd=str(project_dir),
    text=True,
    capture_output=True,
)
print(test_result.stdout)
print(test_result.stderr)
if test_result.returncode != 0:
    raise RuntimeError("A project test failed. Read the output above before launching.")
print("All offline project tests passed.")

## 9. Launch the Streamlit application in Colab

This is the previous Streamlit workflow. It starts the app from the actual project directory, checks that port 8501 responds, and embeds the app in this notebook. Keep the Colab runtime active while using the app.

In [ ]:
import os
import sys
import time
import subprocess
import urllib.request
from pathlib import Path
from google.colab import output

project_dir = Path("/content/AI_Commentator_Studio")
app_file = project_dir / "app.py"
if not app_file.exists():
    raise FileNotFoundError(f"Streamlit app not found: {app_file}")

# Stop an earlier process created in this notebook session, if it is still active.
old_process = globals().get("streamlit_process")
if old_process is not None and old_process.poll() is None:
    old_process.terminate()
    try:
        old_process.wait(timeout=5)
    except subprocess.TimeoutExpired:
        old_process.kill()

PORT = 8501
streamlit_log_path = Path("/content/ai_commentator_streamlit.log")
streamlit_log = open(streamlit_log_path, "w", encoding="utf-8")
streamlit_process = subprocess.Popen(
    [
        sys.executable, "-m", "streamlit", "run", str(app_file),
        "--server.port", str(PORT),
        "--server.address", "0.0.0.0",
        "--server.headless", "true",
        "--browser.gatherUsageStats", "false",
    ],
    cwd=str(project_dir),
    stdout=streamlit_log,
    stderr=subprocess.STDOUT,
)

# Wait for the HTTP server to become ready, not just for the process to exist.
ready = False
for _ in range(30):
    if streamlit_process.poll() is not None:
        break
    try:
        with urllib.request.urlopen(f"http://127.0.0.1:{PORT}/_stcore/health", timeout=2) as resp:
            if resp.status == 200:
                ready = True
                break
    except Exception:
        time.sleep(1)

if not ready:
    streamlit_log.flush()
    streamlit_log.close()
    log_text = streamlit_log_path.read_text(encoding="utf-8", errors="replace")
    print(log_text[-10000:])
    raise RuntimeError("Streamlit did not become healthy. See the log above.")

print("Streamlit is healthy on port", PORT)
print("The app is embedded below. Keep this Colab runtime active.")
output.serve_kernel_port_as_iframe(PORT, height=1050, width="100%")

## 10. Optional: create a temporary public preview link

Use this only when you need to share the running Streamlit app with someone outside your Colab session. It uses a Cloudflare Quick Tunnel; no Cloudflare account or password is required. The URL is temporary, changes between sessions, and stops when the tunnel or Colab runtime stops. It is **not permanent production hosting**. For an in-notebook preview, skip this section.

In [ ]:
# Install Cloudflare's tunnel client in this Colab runtime.
# This does not change the Streamlit app or add any API keys.
import subprocess
import time
import re
from pathlib import Path

cloudflared_deb = "/tmp/cloudflared-linux-amd64.deb"
subprocess.run(
    ["bash", "-lc", f"wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb -O {cloudflared_deb} && dpkg -i {cloudflared_deb}"],
    check=True,
)

# Stop an earlier quick tunnel from this notebook session.
old_tunnel = globals().get("cloudflared_process")
if old_tunnel is not None and old_tunnel.poll() is None:
    old_tunnel.terminate()
    try:
        old_tunnel.wait(timeout=5)
    except subprocess.TimeoutExpired:
        old_tunnel.kill()

tunnel_log_path = Path("/content/cloudflared_tunnel.log")
tunnel_log = open(tunnel_log_path, "w", encoding="utf-8")
cloudflared_process = subprocess.Popen(
    ["cloudflared", "tunnel", "--url", "http://127.0.0.1:8501", "--no-autoupdate"],
    stdout=tunnel_log,
    stderr=subprocess.STDOUT,
)

public_url = None
for _ in range(40):
    time.sleep(1)
    tunnel_log.flush()
    log_text = tunnel_log_path.read_text(encoding="utf-8", errors="replace")
    match = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", log_text)
    if match:
        public_url = match.group(0)
        break
    if cloudflared_process.poll() is not None:
        break

if public_url:
    print("Temporary public preview URL:", public_url)
    print("Keep both Streamlit and cloudflared processes running.")
else:
    print(tunnel_log_path.read_text(encoding="utf-8", errors="replace")[-6000:])
    raise RuntimeError("Could not obtain a Quick Tunnel URL. The local Colab iframe can still be used.")

## 11. Student customization and mini competition

1. Improve Energetic delivery so a confirmed six starts with a punchy celebration and ends with a strong broadcast finish.
2. Make Dramatic delivery build suspense before the confirmed event.
3. Write a Funny version using clever sports humour without mocking players.
4. Compare Urdu and Roman Urdu narration with the same voice.
5. Add one new safe offline fallback line for a football event.

**Debugging exercise:** Temporarily rename `GROQ_API_KEY` in Colab Secrets, run the sample generator, and confirm the app clearly marks the offline fallback. Restore the correct secret name afterward.

**Mini competition:** Students submit the most engaging fact-grounded commentary for one Cricket and one Football scenario. Judge clarity, style difference, language quality, voice performance and factual accuracy.

**Workshop feedback:** [Complete the feedback form](https://forms.gle/DyTZcpnkh8Ax8SG17).
